# Module 08: Pipeline Summary — End-to-End Results

| | |
|---|---|
| **Project** | Face Verification Pipeline — AgeDB-30 / LFW |
| **Notebook** | `notebooks/08_summary.ipynb` |
| **Date** | 2026-03-22 |
| **Dependencies** | Modules 01–07 (all prior notebooks) |
| **Input** | `logs/detection_log_*.csv`, `results/distances_*.csv`, `results/evaluation_metrics.csv`, `results/embedding_metrics.csv` |
| **Output** | Consolidated summary tables and figures |

---

## Objective

This notebook consolidates results from all pipeline stages into a single executive summary:

1. **Stage 1 — Detection & Alignment** (Module 01): face detection rates, alignment quality
2. **Stage 2 — Embedding Extraction** (Module 02): L2-norm statistics, embedding separability
3. **Stage 3 — Verification** (Modules 03–04): ROC/AUC/EER for ArcFace and GhostFaceNetV2
4. **Stage 4 — SwinFace (SOTA ViT)** (Module 06): Transformer-based verification metrics
5. **Stage 5 — PAD / Anti-Spoofing** (Module 07): ISO 30107-3 compliance metrics
6. **Overall ranking** of all models across all datasets

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from sklearn.metrics import roc_curve, auc as sklearn_auc
import warnings
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR  = PROJECT_ROOT / "results"
LOGS_DIR     = PROJECT_ROOT / "logs"
OUTPUT_DIR   = Path("summary_output")
OUTPUT_DIR.mkdir(exist_ok=True)

DATASETS = ["agedb", "lfw"]
DS_LABELS = {"agedb": "AgeDB-30", "lfw": "LFW"}

MODELS = ["arcface", "ghostface", "swinface"]
MODEL_LABELS = {
    "arcface":   "ArcFace (ResNet-50)",
    "ghostface": "GhostFaceNetV2",
    "swinface":  "SwinFace (Swin-T)",
}
MODEL_COLORS = {
    "arcface":   "#1976D2",
    "ghostface": "#D32F2F",
    "swinface":  "#388E3C",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": 120,
})

print(f"Project root : {PROJECT_ROOT}")
print(f"Output dir   : {OUTPUT_DIR.resolve()}")

---
## 1. Stage 1 — Detection & Alignment Summary

Source: `logs/detection_log_{dataset}.csv`

In [ ]:
det_logs = {}
for ds in DATASETS:
    path = LOGS_DIR / f"detection_log_{ds}.csv"
    det_logs[ds] = pd.read_csv(path)

print("Table 1. Detection & Alignment Summary\n")
rows = []
for ds in DATASETS:
    df = det_logs[ds]
    n_total    = len(df)
    n_detected = int(df["face_detected"].sum())
    n_dup      = int(df["is_duplicate"].sum())
    n_unique   = n_total - n_dup
    det_rate   = n_detected / n_total * 100

    non_dup = df[(~df["is_duplicate"]) & (df["face_detected"])]
    tilt_mean_before = non_dup["tilt_before"].abs().mean()
    tilt_mean_after  = non_dup["tilt_after"].abs().mean()
    ipd_mean_after   = non_dup["ipd_after"].mean()

    rows.append({
        "Dataset":          DS_LABELS[ds],
        "Total images":     n_total,
        "Detected (%)": f"{n_detected} ({det_rate:.1f}%)",
        "Unique / Dups":    f"{n_unique} / {n_dup}",
        "Tilt before (°)": f"{tilt_mean_before:.2f}",
        "Tilt after (°)":  f"{tilt_mean_after:.2f}",
        "IPD after (px)":  f"{ipd_mean_after:.1f}",
    })

det_summary = pd.DataFrame(rows)
print(det_summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Fig. 1  Alignment Quality: Tilt Correction Distribution", fontsize=13, fontweight="bold")

colors = {"agedb": MODEL_COLORS["arcface"], "lfw": MODEL_COLORS["ghostface"]}

for ax, ds in zip(axes, DATASETS):
    df = det_logs[ds]
    non_dup = df[(~df["is_duplicate"]) & (df["face_detected"])].copy()
    correction = (non_dup["tilt_before"].abs() - non_dup["tilt_after"].abs())

    ax.hist(correction, bins=50, color=colors[ds], alpha=0.8, edgecolor="white", linewidth=0.4)
    ax.axvline(correction.mean(), color="#212121", lw=1.5, linestyle="--",
               label=f"mean = {correction.mean():.2f}°")
    ax.set_title(DS_LABELS[ds], fontsize=11, fontweight="bold")
    ax.set_xlabel("Tilt correction (°)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "01_alignment_tilt.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 01_alignment_tilt.png")

---
## 2. Stage 2 — Embedding Extraction Summary

Source: `results/embedding_metrics.csv`

In [ ]:
emb_metrics = pd.read_csv(RESULTS_DIR / "embedding_metrics.csv")

print("Table 2. Embedding Metrics (ArcFace & GhostFaceNetV2)\n")
print(emb_metrics.to_string(index=False))

---
## 3. Stage 3–4 — Verification: Load All Distance CSVs

Sources: `results/distances_{model}_{dataset}.csv` — all 6 files (3 models × 2 datasets)

In [ ]:
score_data = {}  # (model, dataset) -> DataFrame

for m in MODELS:
    for ds in DATASETS:
        path = RESULTS_DIR / f"distances_{m}_{ds}.csv"
        if not path.exists():
            print(f"  [MISSING] {path.name}")
            continue
        df = pd.read_csv(path)
        score_data[(m, ds)] = df

print("Table 3. Loaded Distance Files\n")
rows = []
for (m, ds), df in score_data.items():
    n_g = int(df["true_label"].sum())
    n_i = len(df) - n_g
    rows.append({
        "Model":    MODEL_LABELS[m],
        "Dataset":  DS_LABELS[ds],
        "Pairs":    len(df),
        "Genuine":  n_g,
        "Impostor": n_i,
        "Sim mean (gen)": f"{df.loc[df.true_label==1,'cosine_similarity'].mean():.4f}",
        "Sim mean (imp)": f"{df.loc[df.true_label==0,'cosine_similarity'].mean():.4f}",
    })

print(pd.DataFrame(rows).to_string(index=False))

---
## 4. ROC / AUC / EER — All Models × All Datasets

In [ ]:
def compute_eer(fpr, tpr):
    """Equal Error Rate from ROC curve."""
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[idx] + fnr[idx]) / 2.0), float(1.0 - tpr[idx])  # EER, threshold

def tar_at_far(fpr, tpr, target_far):
    """TAR (True Accept Rate) at a given FAR operating point."""
    mask = np.isfinite(fpr) & np.isfinite(tpr)
    fpr_c, tpr_c = fpr[mask], tpr[mask]
    idx = np.searchsorted(fpr_c, target_far)
    if idx == 0:
        return float(tpr_c[0])
    if idx >= len(fpr_c):
        return float(tpr_c[-1])
    return float(tpr_c[idx])

roc_data = {}
metrics  = []

FAR_TARGETS = [1e-2, 1e-3, 1e-4]

for m in MODELS:
    for ds in DATASETS:
        if (m, ds) not in score_data:
            continue
        df = score_data[(m, ds)]
        y_true = df["true_label"].values
        scores = df["cosine_similarity"].values

        fpr, tpr, thresholds = roc_curve(y_true, scores)
        roc_auc = sklearn_auc(fpr, tpr)
        eer, _ = compute_eer(fpr, tpr)
        roc_data[(m, ds)] = {"fpr": fpr, "tpr": tpr, "auc": roc_auc}

        row = {
            "Model":   MODEL_LABELS[m],
            "Dataset": DS_LABELS[ds],
            "AUC":     round(roc_auc, 4),
            "EER (%)": round(eer * 100, 2),
        }
        for far in FAR_TARGETS:
            row[f"TAR@FAR={far:.0e}"] = f"{tar_at_far(fpr, tpr, far)*100:.2f}%"
        metrics.append(row)

metrics_df = pd.DataFrame(metrics)

print("Table 4. Verification Metrics — All Models × All Datasets\n")
print(metrics_df.to_string(index=False))

metrics_df.to_csv(OUTPUT_DIR / "08_all_models_metrics.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR / '08_all_models_metrics.csv'}")

---
## 5. Fig. 2 — ROC Curves: All Models × Both Datasets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Fig. 2  ROC Curves — All Models × Both Datasets", fontsize=13, fontweight="bold")

for ax, ds in zip(axes, DATASETS):
    ax.plot([0, 1], [0, 1], "--", color="#bdbdbd", lw=1, label="Random (AUC=0.50)")
    for m in MODELS:
        if (m, ds) not in roc_data:
            continue
        rd = roc_data[(m, ds)]
        mask = np.isfinite(rd["fpr"]) & np.isfinite(rd["tpr"])
        ax.plot(
            rd["fpr"][mask], rd["tpr"][mask],
            color=MODEL_COLORS[m], lw=2,
            label=f"{MODEL_LABELS[m]}  AUC={rd['auc']:.4f}"
        )
    ax.set_title(DS_LABELS[ds], fontsize=11, fontweight="bold")
    ax.set_xlabel("False Accept Rate (FAR)")
    ax.set_ylabel("True Accept Rate (TAR)")
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.legend(fontsize=8, loc="lower right")

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "02_roc_all_models.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 02_roc_all_models.png")

---
## 6. Fig. 3 — ROC log-scale (FAR ∈ [10⁻⁴, 1])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Fig. 3  ROC Curves (log-scale FAR) — All Models", fontsize=13, fontweight="bold")

for ax, ds in zip(axes, DATASETS):
    for far_t, ls in zip(FAR_TARGETS, ["--", "-.", ":"]):
        ax.axvline(far_t, color="#9e9e9e", lw=0.8, linestyle=ls, alpha=0.7)

    for m in MODELS:
        if (m, ds) not in roc_data:
            continue
        rd = roc_data[(m, ds)]
        mask = np.isfinite(rd["fpr"]) & np.isfinite(rd["tpr"]) & (rd["fpr"] > 0)
        ax.semilogx(
            rd["fpr"][mask], rd["tpr"][mask],
            color=MODEL_COLORS[m], lw=2,
            label=MODEL_LABELS[m]
        )

    ax.set_title(DS_LABELS[ds], fontsize=11, fontweight="bold")
    ax.set_xlabel("FAR (log scale)")
    ax.set_ylabel("TAR")
    ax.set_xlim([1e-4, 1.0])
    ax.set_ylim([0.0, 1.01])
    ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "03_roc_logscale.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 03_roc_logscale.png")

---
## 7. Fig. 4 — Score Distributions: Genuine vs. Impostor

In [ ]:
fig, axes = plt.subplots(len(MODELS), 2, figsize=(13, 4 * len(MODELS)))
fig.suptitle(
    "Fig. 4  Score Distributions — Genuine vs. Impostor Cosine Similarity",
    fontsize=13, fontweight="bold", y=1.01
)

for row_idx, m in enumerate(MODELS):
    for col_idx, ds in enumerate(DATASETS):
        ax = axes[row_idx][col_idx]
        if (m, ds) not in score_data:
            ax.set_visible(False)
            continue
        df = score_data[(m, ds)]
        gen = df.loc[df["true_label"] == 1, "cosine_similarity"]
        imp = df.loc[df["true_label"] == 0, "cosine_similarity"]

        ax.hist(gen, bins=60, alpha=0.75, color="#1976D2", label=f"Genuine (n={len(gen)})")
        ax.hist(imp, bins=60, alpha=0.75, color="#D32F2F", label=f"Impostor (n={len(imp)})")
        ax.set_title(f"{MODEL_LABELS[m]} / {DS_LABELS[ds]}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Cosine Similarity")
        ax.set_ylabel("Count")
        ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "04_score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 04_score_distributions.png")

---
## 8. Fig. 5 — TAR @ FAR Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Fig. 5  TAR@FAR Operating Points — All Models × Both Datasets",
    fontsize=13, fontweight="bold"
)

x = np.arange(len(MODELS))
width = 0.22
far_colors = ["#1565C0", "#6A1B9A", "#B71C1C"]
far_labels = ["FAR=1%", "FAR=0.1%", "FAR=0.01%"]

for ax, ds in zip(axes, DATASETS):
    for i, (far, fc, fl) in enumerate(zip(FAR_TARGETS, far_colors, far_labels)):
        vals = []
        for m in MODELS:
            if (m, ds) not in roc_data:
                vals.append(0.0)
                continue
            rd = roc_data[(m, ds)]
            vals.append(tar_at_far(rd["fpr"], rd["tpr"], far) * 100)

        bars = ax.bar(x + (i - 1) * width, vals, width, label=fl,
                      color=fc, alpha=0.85, edgecolor="white", linewidth=0.5)
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{val:.1f}", ha="center", va="bottom", fontsize=7, fontweight="bold"
            )

    ax.set_title(DS_LABELS[ds], fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODELS], fontsize=8)
    ax.set_ylabel("TAR (%)")
    ax.set_ylim([0, 108])
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "05_tar_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 05_tar_bar.png")

---
## 9. Fig. 6 — AUC & EER Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Fig. 6  AUC and EER — All Models × Both Datasets", fontsize=13, fontweight="bold")

x = np.arange(len(DATASETS))
width = 0.25

for ax_idx, (ax, metric, ylabel, higher_is_better) in enumerate(zip(
    axes,
    ["AUC",   "EER (%)"],
    ["AUC",   "EER (%) — lower is better"],
    [True,    False]
)):
    for i, m in enumerate(MODELS):
        vals = []
        for ds in DATASETS:
            row = metrics_df[(metrics_df["Model"] == MODEL_LABELS[m]) &
                             (metrics_df["Dataset"] == DS_LABELS[ds])]
            vals.append(float(row[metric].values[0]) if len(row) > 0 else 0.0)

        bars = ax.bar(x + (i - 1) * width, vals, width,
                      label=MODEL_LABELS[m],
                      color=MODEL_COLORS[m], alpha=0.85,
                      edgecolor="white", linewidth=0.5)
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + (0.0003 if metric == "AUC" else 0.05),
                f"{val:.4f}" if metric == "AUC" else f"{val:.2f}%",
                ha="center", va="bottom", fontsize=7.5, fontweight="bold"
            )

    ax.set_xticks(x)
    ax.set_xticklabels([DS_LABELS[ds] for ds in DATASETS], fontsize=10)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.set_title(metric, fontsize=11, fontweight="bold")
    if metric == "AUC":
        ax.set_ylim([0.97, 1.002])

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "06_auc_eer.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 06_auc_eer.png")

---
## 10. Stage 5 — PAD / Anti-Spoofing Summary (Module 07)

Metrics reported from notebook 07 output (hardcoded from executed cells — `pad_output/` is gitignored).

In [ ]:
# ── Try to load from pad_output/ first; fall back to values from nb07 outputs ──
pad_output_dir = PROJECT_ROOT / "notebooks" / "pad_output"

pad_summary_path   = pad_output_dir / "pad_summary_metrics.csv"
per_attack_path    = pad_output_dir / "per_attack_apcer.csv"

if pad_summary_path.exists():
    pad_summary_df = pd.read_csv(pad_summary_path)
    print("PAD metrics loaded from pad_output/pad_summary_metrics.csv")
else:
    # Values from Module 07 executed output cells
    pad_summary_df = pd.DataFrame({
        "Metric": ["Accuracy", "AUC (ROC)", "APCER", "BPCER", "ACER"],
        "Value":  [0.9544,     0.9792,      0.1250,  0.0192,  0.0721],
        "Target (iBeta L2)": ["—", "—", "≤0.05", "—", "≤0.10"],
    })
    print("pad_output/ not found — using values from Module 07 notebook output.")

if per_attack_path.exists():
    per_attack_df = pd.read_csv(per_attack_path)
    print("Per-attack APCER loaded from pad_output/per_attack_apcer.csv")
else:
    per_attack_df = pd.DataFrame({
        "Attack Type":  ["Print", "Screen Replay", "Paper Mask", "OVERALL (max)"],
        "N attacks":    [400, 400, 400, 1200],
        "FP (accepted)":[23, 0, 0, 23],
        "APCER":        [0.0575, 0.0, 0.0, 0.0400],
    })
    print("per_attack_apcer.csv not found — using values from Module 07 notebook output.")

print("\nTable 5. PAD Summary Metrics (ISO 30107-3)\n")
print(pad_summary_df.to_string(index=False))
print("\nTable 6. Per-Attack-Type APCER\n")
print(per_attack_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Fig. 7  PAD Anti-Spoofing Performance (ISO 30107-3)", fontsize=13, fontweight="bold")

# Panel 1: summary bar
metric_names = pad_summary_df["Metric"].tolist()
metric_vals  = pad_summary_df["Value"].tolist()
bar_colors   = ["#1976D2" if v >= 0.9 else "#D32F2F" if v >= 0.1 else "#388E3C"
                for v in metric_vals]

bars = axes[0].barh(metric_names, metric_vals, color=bar_colors, alpha=0.85,
                    edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, metric_vals):
    axes[0].text(
        val + 0.005, bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}", va="center", fontsize=9, fontweight="bold"
    )
axes[0].set_xlim([0, 1.12])
axes[0].set_title("PAD Metrics", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Value")
axes[0].invert_yaxis()

# Panel 2: per-attack APCER
attack_types = per_attack_df.iloc[:, 0].tolist()
apcer_vals   = per_attack_df["APCER"].tolist()
a_colors     = ["#D32F2F" if v > 0.05 else "#388E3C" for v in apcer_vals]

bars2 = axes[1].bar(attack_types, apcer_vals, color=a_colors, alpha=0.85,
                    edgecolor="white", linewidth=0.5)
axes[1].axhline(0.05, color="#F57F17", lw=1.5, linestyle="--", label="iBeta L2 limit (0.05)")
for bar, val in zip(bars2, apcer_vals):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
        f"{val:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold"
    )
axes[1].set_title("Per-Attack-Type APCER", fontsize=11, fontweight="bold")
axes[1].set_ylabel("APCER")
axes[1].set_ylim([0, 0.10])
axes[1].legend(fontsize=8)
axes[1].tick_params(axis="x", labelrotation=15)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "07_pad_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 07_pad_summary.png")

---
## 11. Final Rankings — Best Model per Dataset

In [ ]:
print("Table 7. Final Rankings — Sorted by AUC (descending)\n")

for ds in DATASETS:
    sub = metrics_df[metrics_df["Dataset"] == DS_LABELS[ds]].copy()
    sub = sub.sort_values("AUC", ascending=False).reset_index(drop=True)
    sub.insert(0, "Rank", sub.index + 1)
    print(f"  {DS_LABELS[ds]}")
    print(sub.to_string(index=False))
    print()

print("Table 8. Final Rankings — Sorted by EER (ascending)\n")
for ds in DATASETS:
    sub = metrics_df[metrics_df["Dataset"] == DS_LABELS[ds]].copy()
    sub = sub.sort_values("EER (%)", ascending=True).reset_index(drop=True)
    sub.insert(0, "Rank", sub.index + 1)
    print(f"  {DS_LABELS[ds]}")
    print(sub.to_string(index=False))
    print()

In [ ]:
# ── Average rank across both datasets ─────────────────────────────────────────
print("Table 9. Average AUC & EER across both datasets\n")

avg_rows = []
for m in MODELS:
    sub = metrics_df[metrics_df["Model"] == MODEL_LABELS[m]]
    if sub.empty:
        continue
    avg_rows.append({
        "Model":       MODEL_LABELS[m],
        "AUC (avg)":   round(sub["AUC"].mean(), 4),
        "EER% (avg)":  round(sub["EER (%)"].mean(), 3),
    })

avg_df = pd.DataFrame(avg_rows).sort_values("AUC (avg)", ascending=False).reset_index(drop=True)
avg_df.insert(0, "Overall Rank", avg_df.index + 1)
print(avg_df.to_string(index=False))

---
## 12. Output File Summary

In [ ]:
output_files = sorted(list(OUTPUT_DIR.glob("*.png")) + list(OUTPUT_DIR.glob("*.csv")))

print(f"Module 08 output files ({len(output_files)} files):\n")
for f in output_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} {size_kb:>7.1f} KB")

print("\n" + "═" * 60)
print(" PIPELINE COMPLETE")
print("═" * 60)
print(f"  Modules executed : 01 → 08")
print(f"  Models evaluated : {', '.join(MODEL_LABELS[m] for m in MODELS)}")
print(f"  Datasets         : AgeDB-30, LFW")
print(f"  Total pairs      : {sum(len(score_data[(m,ds)]) for m in MODELS for ds in DATASETS if (m,ds) in score_data):,}")
print("═" * 60)